# 🚲 ValenWheel — Train the Valenbisi models (Google Colab)

Trains the forecasting + stockout models used by the **ValenWheel** Streamlit app and produces every artifact it needs:

* `models/forecast_model.pkl` — regressor → bikes available
* `models/stockout_model.pkl` — classifier → P(no bikes)
* `models/model_meta.json` — leaderboard + metrics
* `data/station_profiles.parquet`, `data/hourly_profiles.parquet`

**Data:** historical 15-min Valenbisi snapshots from [github.com/ceferra/valenbici](https://github.com/ceferra/valenbici) (Valencia open data) + Open-Meteo weather.

**Pipeline:** parse → feature engineering (cyclical time, geo, weather) → time-ordered model selection (4 regressors) → stockout classifier → SHAP → KMeans station clustering.

> Run **Runtime ▸ Run all**.

## 1 · Install dependencies

In [ ]:
!pip -q install scikit-learn==1.7.2 shap==0.49.1 pyarrow pandas numpy joblib

## 2 · Get the project code
Clone your repo so training uses the **exact** same `data_prep.py` as the app (no train/serve skew). Replace the URL with your fork.

In [ ]:
REPO_URL = "https://github.com/<your-user>/valenbisi.git"  # <-- EDIT ME
import os
if not os.path.exists('valenbisi'):
    !git clone $REPO_URL valenbisi
%cd valenbisi

## 3 · Download the open data (snapshots + weather)
Samples several weeks across seasons from `ceferra/valenbici` and joins Open-Meteo weather. Use `--smoke` for a fast 3-day test.

In [ ]:
!python download_data.py

## 4 · Train — model selection, classifier, clustering

In [ ]:
!python train.py

## 5 · Inspect the results

In [ ]:
import json, pandas as pd
meta = json.load(open('models/model_meta.json'))
print('Best regressor:', meta['best_regressor'], '| test:', meta['regressor_test'])
print('Stockout classifier test:', meta['stockout_test'])
print('Clustering:', meta['clustering'])
pd.DataFrame(meta['regressor_leaderboard']).sort_values('mae_bikes')

In [ ]:
# SHAP global importance for the forecasting model
import sys, joblib, numpy as np, pandas as pd, shap
sys.path.insert(0, 'app')
import data_prep as dp
reg = joblib.load('models/forecast_model.pkl')
hist = pd.read_parquet('data/history.parquet')
X = hist[dp.all_feature_names()].sample(2000, random_state=0)
Xt = reg.named_steps['pre'].transform(X)
expl = shap.TreeExplainer(reg.named_steps['model'])
shap.summary_plot(expl.shap_values(Xt), Xt,
                  feature_names=reg.named_steps['pre'].get_feature_names_out(),
                  max_display=12)

## 6 · Download the trained artifacts
Drop `models/*` and `data/*.parquet` back into your repo, then redeploy the Streamlit app.

In [ ]:
import shutil
shutil.make_archive('valenwheel_models', 'zip', '.', 'models')
shutil.make_archive('valenwheel_data', 'zip', 'data', '.')
try:
    from google.colab import files
    files.download('valenwheel_models.zip')
    files.download('valenwheel_data.zip')
except Exception as e:
    print('Not in Colab; artifacts saved locally.', e)